# Flavor Expansion Walkthrough


In [2]:
import sys
from pathlib import Path
from fractions import Fraction

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from symbolica import S
from symbolica.community.spenso import Representation

from model import (
    COLOR_FUND_INDEX,
    Field,
    GaugeGroup,
    GaugeRepresentation,
    IndexType,
    Model,
    Parameter,
    SPINOR_INDEX,
    flavor_index,
)
from symbolic.spenso_structures import gauge_generator, structure_constant

print("Python:", sys.executable)
print("Repository root:", ROOT)


In [3]:
def show(title, result):
  """Print vertex signatures and rules in a notebook-friendly format."""
  print(title)
  if isinstance(result, dict):
    print(f"{len(result)} vertex signature(s)\n")
    for signature, expression in result.items():
      print("Vertex:", signature)
      print("Rule:", expression)
      print()
  else:
    print(result)
    print()


def off_diagonal_zero_components(index):
  """Build {(i,j): 0} for all i != j — useful for diagonal Yukawa matrices."""
  return {
    (row, col): 0
    for row in range(1, index.dimension + 1)
    for col in range(1, index.dimension + 1)
    if row != col
  }


## Example 1 — Charged leptons


In [5]:
Generation = flavor_index("Generation", 3, prefix="f")

l = Field(
    "l",
    spin=Fraction(1, 2),
    self_conjugate=False,
    indices=(Generation, SPINOR_INDEX),
    symbol=S("l"),
    conjugate_symbol=S("lbar"),
    quantum_numbers={"Q": -1, "LeptonNumber": 1},
    flavor_index=Generation,
    class_members=("e", "mu", "ta"),
)
e, mu, ta = l.class_members
Phi = Field("Phi", spin=0, self_conjugate=True, symbol=S("Phi"))

print("flavor.is_flavor:", Generation.is_flavor)
print("flavor.dimension:", Generation.dimension)
print("l.indices:", [index.name for index in l.indices])
print("l.flavor_index_slot():", l.flavor_index_slot())
print("members:", [member.name for member in l.class_members])
print("member indices:", {member.name: [index.name for index in member.indices] for member in l.class_members})
print("mapping:", {k: l.class_member_for(k).name for k in range(1, Generation.dimension + 1)})


f = S("f")
lepton_model = Model(
    name="Charged leptons",
    fields=(l, Phi),
    lagrangian_decl=S("lam") * l.bar(f) * l(f) * Phi,
)

print("\ncompact signatures:", [s.names for s in lepton_model.vertex_signatures(flavor_expand=False)])
print("expanded signatures:", [s.names for s in lepton_model.vertex_signatures(flavor_expand=True)])


In [6]:
print("compact rule l.bar, l, Phi")
print(lepton_model.feynman_rule(l.bar, l, Phi, simplify=True))
print()
show("expanded rules for all members:", lepton_model.feynman_rules(flavor_expand=True,simplify=True))
print()
print("off-diagonal e.bar, mu, Phi is absent")
try:
    print(lepton_model.feynman_rule(e.bar, mu, Phi, simplify=True, flavor_expand=True))
except Exception as exc:
    print(type(exc).__name__ + ":", exc)


In [7]:

Colour = COLOR_FUND_INDEX
fq, color = S("fq", "c")

q = Field(
    "q",
    spin=Fraction(1, 2),
    self_conjugate=False,
    indices=(Generation, Colour, SPINOR_INDEX),
    symbol=S("q"),
    conjugate_symbol=S("qbar"),
    flavor_index=Generation,
    class_members=("d", "s", "b"),
)
d, s, b = q.class_members
PhiQ = Field("PhiQ", spin=0, self_conjugate=True, symbol=S("PhiQ"))

color_model = Model(
    fields=(q, PhiQ),
    lagrangian_decl=S("gQ") * q.bar(fq, color) * q(fq, color) * PhiQ,
)

print()
print("same flavor expansion with a spectator color index")
print("colored member indices:", {member.name: [index.name for index in member.indices] for member in q.class_members})
print("expanded signatures:", [s.names for s in color_model.vertex_signatures(flavor_expand=True)])
show("\n compact rules:", color_model.feynman_rules(simplify=True))
show("expanded rules for all members:", color_model.feynman_rules(flavor_expand=True, simplify=True))


## Example 2 — diagonal Yukawa



In [9]:
Colour = COLOR_FUND_INDEX

colour_fund = GaugeRepresentation(
    index=Colour,
    generator_builder=gauge_generator,
    name="fund",
)



uq = Field(
    "uq",
    spin=Fraction(1, 2),
    self_conjugate=False,
    indices=(Generation, Colour, SPINOR_INDEX),
    symbol=S("uq"),
    conjugate_symbol=S("uqbar"),
    quantum_numbers={"Q": Fraction(2, 3)},
    flavor_index=Generation,
    class_members=("u", "c", "t"),
)
u, c, t = uq.class_members
dq = Field(
    "dq",
    spin=Fraction(1, 2),
    self_conjugate=False,
    indices=(Generation, Colour, SPINOR_INDEX),
    symbol=S("dq"),
    conjugate_symbol=S("dqbar"),
    quantum_numbers={"Q": Fraction(-1, 3)},
    flavor_index=Generation,
    class_members=("d", "s", "b"),
)
d, s, b = dq.class_members

gQ = Parameter("gQ")
yu = Parameter(
    "yu",
    indices=(Generation, Generation),
    components=off_diagonal_zero_components(Generation),
)

f, h, col = S("f", "h", "col")

quark_model = Model(
    name="Quark Yukawa demo",
    fields=(uq, dq, Phi),
    parameters=(gQ, yu),
    lagrangian_decl=(
        gQ * uq.bar(f) * uq(f) * Phi
        + yu(f, h) * uq.bar(f) * uq(h) * Phi
    ),
)

print("compact signatures:")
for signature in quark_model.vertex_signatures(flavor_expand=False):
    print(" ", signature.names)

print("\nexpanded signatures (Generation):")
for signature in quark_model.vertex_signatures(flavor_expand=True):
    print(" ", signature.names)

show("\nexpanded rules for all members:", quark_model.feynman_rules(flavor_expand=True, simplify=True))

print("\noff-diagonal u.bar, c, Phi removed by components={(i,j): 0}:")
try:
    print(quark_model.feynman_rule(u.bar, c, Phi, simplify=True, flavor_expand=Generation))
except Exception as exc:
    print(type(exc).__name__ + ":", exc)


## Example 3 — Non-diagonal two-index flavor tensor



In [11]:
f_mix, g_mix = S("fMix", "gMix")
Y = Parameter("Y", indices=(Generation, Generation))

mixed_model = Model(
    name="Lepton mixing",
    fields=(l, Phi),
    parameters=(Y,),
    lagrangian_decl=S("gY") * l.bar(f_mix) * Y(f_mix, g_mix) * l(g_mix) * Phi,
)

print("expanded signatures for Y[f, g]:")
print([s.names for s in mixed_model.vertex_signatures(flavor_expand=True)])
print()
print("e.bar, mu, Phi")
print(mixed_model.feynman_rule(e.bar, mu, Phi, simplify=True, flavor_expand=True))
print()
print("mu.bar, e, Phi")
print(mixed_model.feynman_rule(mu.bar, e, Phi, simplify=True, flavor_expand=True))
print()
print("ta.bar, ta, Phi")
print(mixed_model.feynman_rule(ta.bar, ta, Phi, simplify=True, flavor_expand=True))


## Example 4 — One-index diagonal tensor and default summation

A single-index flavor parameter such as `y(f)` in `y(f) * l.bar(f) * l(f) * Phi` allows the same label to appear more than twice in one term by default. Set `allow_summation=False` on the parameter to reject that pattern.


In [13]:
y_diag = Parameter("yDiag", indices=(Generation,))
diag_model = Model(
    fields=(l, Phi),
    parameters=(y_diag,),
    lagrangian_decl=y_diag(f) * l.bar(f) * l(f) * Phi,
)

print("diagonal y[f] signatures (default summation):", [s.names for s in diag_model.vertex_signatures(flavor_expand=True)])
print()
show("Total vertex", diag_model.feynman_rule(simplify=True, flavor_expand=True))
print()

y_opt_out = Parameter("yOptOut", indices=(Generation,), allow_summation=False)
opt_out_model = Model(
    fields=(l, Phi),
    parameters=(y_opt_out,),
    lagrangian_decl=y_opt_out(f) * l.bar(f) * l(f) * Phi,
)

print("with allow_summation=False:")
try:
    opt_out_model.vertex_signatures(flavor_expand=True)
except Exception as exc:
    print(type(exc).__name__ + ":", exc)


## Example 5 — Zero tensor components are dropped




In [15]:
Ydiag = Parameter(
    "Ydiag",
    indices=(Generation, Generation),
    components=off_diagonal_zero_components(Generation),
)

zero_model = Model(
    fields=(l, Phi),
    parameters=(Ydiag,),
    lagrangian_decl=S("gDiag") * l.bar(f_mix) * Ydiag(f_mix, g_mix) * l(g_mix) * Phi,
)

print("only diagonal signatures remain:", [s.names for s in zero_model.vertex_signatures(flavor_expand=True)])
print()
show("compact Feynman rules", zero_model.feynman_rules(flavor_expand=False))
print("================================================================")
show("expanded Feynman rules", zero_model.feynman_rules(flavor_expand=True))



## Example 6 — Shared flavor label across two field classes

The same `Generation` label synchronizes left- and right-handed lepton classes.


In [17]:
lL = Field(
    "lL",
    spin=Fraction(1, 2),
    self_conjugate=False,
    indices=(Generation, SPINOR_INDEX),
    symbol=S("lL"),
    conjugate_symbol=S("lLbar"),
    flavor_index=Generation,
    class_members=("eL", "muL", "taL"),
)
eL, muL, taL = lL.class_members
lR = Field(
    "lR",
    spin=Fraction(1, 2),
    self_conjugate=False,
    indices=(Generation, SPINOR_INDEX),
    symbol=S("lR"),
    conjugate_symbol=S("lRbar"),
    flavor_index=Generation,
    class_members=("eR", "muR", "taR"),
)
eR, muR, taR = lR.class_members
yLR = Parameter("yLR", indices=(Generation,))

shared_model = Model(
    fields=(lL, lR, Phi),
    parameters=(yLR,),
    lagrangian_decl=yLR(f) * lL.bar(f) * lR(f) * Phi,
)

print("shared flavor label across lL and lR:", [s.names for s in shared_model.vertex_signatures(flavor_expand=True)])
print()
print("taL.bar, taR, Phi")
print(shared_model.feynman_rule(taL.bar, taR, Phi, simplify=True, include_delta=True, flavor_expand=True))
show("expanded vertex", shared_model.feynman_rule( simplify=True, flavor_expand=True))


## Example 7 — Independent flavor labels: CKM-like up–down mixing



In [19]:
W = Field("W", spin=0, self_conjugate=True, symbol=S("W"))
V = Parameter("V", indices=(Generation, Generation))
colour = S("colour")

mixing_model = Model(
    fields=(uq, dq, W),
    parameters=(V,),
    lagrangian_decl=uq.bar(f, colour) * V(f, h) * dq(h, colour) * W,
)

print("independent uq[f] and dq[h] labels:")
print([s.names for s in mixing_model.vertex_signatures(flavor_expand=True)])
print()
show("excepanded vertex", mixing_model.feynman_rules(flavor_expand=True, simplify=True))


## Example 8 — Selected flavor expansion

`flavor_expand` can target one index (`flavor_expand=Generation`) or several (`flavor_expand=(Generation, SU2D)`), instead of expanding every flavor index at once.


In [21]:
SU2D = flavor_index("SU2D", 2, prefix="d")
chi = Field(
    "chi",
    spin=Fraction(1, 2),
    self_conjugate=False,
    indices=(SU2D, SPINOR_INDEX),
    symbol=S("chi"),
    conjugate_symbol=S("chibar"),
    flavor_index=SU2D,
    class_members=("chi1", "chi2"),
)
chi1, chi2 = chi.class_members
d_sel, c_sel = S("d", "c")

selected_model = Model(
    fields=(uq, chi, Phi),
    lagrangian_decl=S("gSel") * uq.bar(f, c_sel) * chi(d_sel) * Phi,
)

print("expand Generation only:")
print([s.names for s in selected_model.vertex_signatures(flavor_expand=Generation)])
print()
print("expand SU2D only:")
print([s.names for s in selected_model.vertex_signatures(flavor_expand=SU2D)])
print()
print("expand both:")
print([s.names for s in selected_model.vertex_signatures(flavor_expand=(Generation, SU2D))])


## Example 9 — Representative validation errors

The metadata layer fails early when declarations are inconsistent.


In [23]:
print("wrong member count")
try:
    Field(
        "BadPsi",
        spin=Fraction(1, 2),
        self_conjugate=False,
        indices=(Generation, SPINOR_INDEX),
        symbol=S("BadPsi"),
        conjugate_symbol=S("BadPsibar"),
        flavor_index=Generation,
        class_members=("e", "mu"),
    )
except Exception as exc:
    print(type(exc).__name__ + ":", exc)
print()

l_no_members = Field(
    "lNoMembers",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("l_no_members"),
    conjugate_symbol=S("l_no_members_bar"),
    indices=(Generation, SPINOR_INDEX),
    flavor_index=Generation,
)
missing_model = Model(
    fields=(l_no_members, Phi),
    lagrangian_decl=S("gMissing") * l_no_members.bar(f) * l_no_members(f) * Phi,
)
print("flavor expansion requested but no class members are defined")
try:
    missing_model.vertex_signatures(flavor_expand=True)
except Exception as exc:
    print(type(exc).__name__ + ":", exc)
print()

plain_index = IndexType(
    "PlainGeneration",
    Representation.cof(3),
    kind="plain_generation",
    dimension=3,
    prefix="p",
)
print("flavor_index must use is_flavor=True")
try:
    Field(
        "WrongFlavorFlag",
        spin=Fraction(1, 2),
        self_conjugate=False,
        symbol=S("wrongflavor"),
        conjugate_symbol=S("wrongflavorbar"),
        indices=(plain_index, SPINOR_INDEX),
        flavor_index=plain_index,
        class_members=(e, mu, ta),
    )
except Exception as exc:
    print(type(exc).__name__ + ":", exc)
